# Promotion forecast lab (Chronos-2, synthetic data)

An explicitly authorized adaptation of the official Chronos-2 quickstart notebook:

- repository: `amazon-science/chronos-forecasting`
- commit: `10afa9ebe016e514f9d7dc1aa873f66af57e116b`
- notebook: `notebooks/chronos-2-quickstart.ipynb`
- source URL: https://github.com/amazon-science/chronos-forecasting/blob/10afa9ebe016e514f9d7dc1aa873f66af57e116b/notebooks/chronos-2-quickstart.ipynb

The upstream notebook demonstrates Chronos-2 covariate forecasting on downloaded Rossmann retail and electricity-price datasets. This notebook keeps the same forecasting method but replaces those datasets with a fully synthetic fixture (no retail or electricity data is downloaded, stored, or redistributed). It runs the Apache-2.0 `autogluon/chronos-2-small` model (pinned revision) on CPU in float32. Outputs are scenario forecasts on synthetic data: not causal estimates and not production-accuracy claims.

Stages: (1) prepare synthetic history and known future covariates, (2) real Chronos-2 inference plus a seasonal-7 baseline, (3) evaluate and persist `results.json` in the current directory.

In [ ]:
import json
import os
import sys

import numpy as np

try:
    import forecast_core as fc
except ImportError:  # run from a fresh directory with PROJECT_ROOT supplied
    project_root = os.environ.get("PROJECT_ROOT")
    if project_root and os.path.isdir(project_root):
        sys.path.insert(0, project_root)
    import forecast_core as fc

print("chronos-forecasting model:", fc.MODEL_ID, "@", fc.MODEL_REVISION)
print("license:", fc.MODEL_LICENSE, "| device: cpu | dtype: float32")

## Stage 1 - Prepare synthetic history and known future covariates

In [ ]:
SEED = 42
HORIZON = 28
PROMOTION_START = 7
PROMOTION_DAYS = 7

data = fc.make_fixture(
    seed=SEED, horizon=HORIZON,
    promotion_start=PROMOTION_START, promotion_days=PROMOTION_DAYS,
)
print("context length:", len(data["context"]))
print("horizon:", len(data["actual"]))
print("promotion days in history:", int(data["past_promotion"].sum()))
print("promotion days in future:", int(data["future_promotion"].sum()))
print("last 5 context values:", np.round(data["context"][-5:], 3))

## Stage 2 - Real Chronos-2 inference and seasonal baseline

Only `data["context"]` is used as the input target; the held-out `actual` is never used for prediction. Past and future covariates (promotion flag, weekly sine) are supplied to the model exactly as in the upstream covariate-forecasting example.

In [ ]:
prediction = fc.predict(data, use_covariates=True)
baseline = fc._seasonal7_baseline(data["context"], HORIZON)
print("forecast head:", np.round(prediction["forecast"][:5], 3))
print("p10 head:", np.round(prediction["lower"][:5], 3))
print("p90 head:", np.round(prediction["upper"][:5], 3))
print("seasonal-7 baseline head:", np.round(baseline[:5], 3))

## Stage 3 - Evaluate and persist results

Metrics: MAE of the Chronos-2 forecast and of the seasonal-7 baseline, plus the observed p10-p90 coverage. The nominal 80% interval is not calibrated on this synthetic fixture, so observed coverage deviating from 80% is expected.

In [ ]:
actual = np.asarray(data["actual"], dtype=np.float64)
forecast = np.asarray(prediction["forecast"], dtype=np.float64)
lower = np.asarray(prediction["lower"], dtype=np.float64)
upper = np.asarray(prediction["upper"], dtype=np.float64)
baseline64 = np.asarray(baseline, dtype=np.float64)

mae = float(np.mean(np.abs(forecast - actual)))
baseline_mae = float(np.mean(np.abs(baseline64 - actual)))
coverage = float(np.mean((actual >= lower) & (actual <= upper)))

results = {
    "mae": mae,
    "baseline_mae": baseline_mae,
    "coverage": coverage,
    "horizon": HORIZON,
    "seed": SEED,
    "model": fc.MODEL_ID,
    "model_revision": fc.MODEL_REVISION,
    "forecast": [float(v) for v in forecast],
}
assert all(np.isfinite(v) for v in (mae, baseline_mae, coverage)), "non-finite metric"

with open("results.json", "w", encoding="utf-8") as fh:
    json.dump({"results": results}, fh, indent=2)

print(f"mae={mae:.4f} baseline_mae={baseline_mae:.4f} coverage={coverage:.2f}")
print("wrote results.json to", os.getcwd())